In [41]:
import os
import pandas as pd
from skimage import transform
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn import metrics
from sklearn.metrics import roc_auc_score, f1_score
from skimage import color

from Multiview_autoencoder.Multiview_autoencoder import MultiviewImageDataset, MultiViewConvAutoencoder, extract_features

In [43]:
# Load multi-view dataset
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

image_tensors = []
image_arrays = []
rgb_imgs = []
gray_imgs = []
number = []
image_folder = './MultiviewImageDataset/cv-mixture' 
for filename in os.listdir(image_folder):
    if filename.endswith('.png'):
        num = filename.split('.')[0]
        num = int(num) - 1
        number.append(num)
        img = Image.open(os.path.join(image_folder, filename))
        img_array = np.array(img)
        if len(img_array.shape) == 3:
            #print(num)
            rgb_image = color.rgba2rgb(img)
            rgb_imgs.append(rgb_image)
            gray_img = color.rgb2gray(rgb_image)
            gray_imgs.append(gray_img)
            img_array = np.array(gray_img)
        image_arrays.append(img_array)
        new_size = (400, 400)
        resized_img = transform.resize(img_array, new_size)
        img_tensor = torch.tensor(resized_img)
        img_tensor = img_tensor.to(torch.float32)
        image_tensors.append(img_tensor)

image_tensors_sub = []
image_arrays_sub = []
rgb_imgs_sub = []
gray_imgs_sub = []
number_sub = []
sub_image_folder = './MultiviewImageDataset/cv-sub' 
for filename in os.listdir(sub_image_folder):
    if filename.endswith('.png'):
        num = filename.split('-')[0]
        num = int(num) - 1
        number_sub.append(num)
        img = Image.open(os.path.join(sub_image_folder, filename))
        img_array = np.array(img)
        if len(img_array.shape) == 3:
            #print(num)
            rgb_image = color.rgba2rgb(img)
            rgb_imgs_sub.append(rgb_image)
            gray_img = color.rgb2gray(rgb_image)
            gray_imgs_sub.append(gray_img)
            img_array = np.array(gray_img)
        image_arrays_sub.append(img_array)
        new_size = (400, 400)
        resized_img = transform.resize(img_array, new_size)
        img_tensor = torch.tensor(resized_img)
        img_tensor = img_tensor.to(torch.float32)
        image_tensors_sub.append(img_tensor)

sub_dict = {}
for idx, sub in zip(number_sub,image_tensors_sub):
    sub_dict[idx] = sub
sub_train = []
for i in number:
    y = sub_dict[i]
    sub_train.append(y)        

image_tensors_cat = []
image_arrays_cat = []
rgb_imgs_cat = []
gray_imgs_cat = []
number_cat = []
cat_image_folder = './MultiviewImageDataset/cv-cat'
for filename in os.listdir(cat_image_folder):
    if filename.endswith('.png'):
        num = filename.split('-')[0]
        num = int(num) - 1
        number_cat.append(num)
        img = Image.open(os.path.join(cat_image_folder, filename))
        img_array = np.array(img)
        if len(img_array.shape) == 3:
            #print(num)
            rgb_image = color.rgba2rgb(img)
            rgb_imgs_cat.append(rgb_image)
            gray_img = color.rgb2gray(rgb_image)
            gray_imgs_cat.append(gray_img)
            img_array = np.array(gray_img)
        image_arrays_sub.append(img_array)
        new_size = (400, 400)
        resized_img = transform.resize(img_array, new_size)
        img_tensor = torch.tensor(resized_img)
        img_tensor = img_tensor.to(torch.float32)
        image_tensors_cat.append(img_tensor)

cat_dict = {}
for idx, cat in zip(number_cat,image_tensors_cat):
    cat_dict[idx] = cat
cat_train = []
for i in number:
    y = cat_dict[i]
    cat_train.append(y)

df = pd.read_csv('./MultiviewImageDataset/data_all.csv')
target = df['tag'].to_list()
target = np.array(target)

num_csv = df['num'].to_list()
num_csv = np.array(num_csv)

target_dict = {}
for idx, y in zip(num_csv, target):
    target_dict[idx] = y

dataset = MultiviewImageDataset(image_tensors, sub_train, cat_train, number)
data_loader = DataLoader(dataset, batch_size=1, shuffle=True)

Using device: mps


In [46]:
# Training autoencoder
model = MultiViewConvAutoencoder()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())
model = model.to(device)

num_train = []
decoding_list1 = []
decoding_list2 = []
decoding_list3 = []
for epoch in range(20):
    stop_training = False
    for img1, img2, img3, num in data_loader:
        img1, img2, img3 = img1.to(device), img2.to(device), img3.to(device)           
        optimizer.zero_grad()
        rec1, rec2, rec3, _ = model(img1, img2, img3)
        img1 = img1.unsqueeze(1)
        img2 = img2.unsqueeze(1)
        img3 = img3.unsqueeze(1)
        loss1 = criterion(rec1, img1)
        loss2 = criterion(rec2, img2)
        loss3 = criterion(rec3, img3)
        loss_total = loss1 + loss2 + loss3  
        loss_total.backward()
        optimizer.step()
        if loss_total.item() < 0.0002:
            stop_training = True
            break 
    print(f'Autoencoder_epoch [{epoch+1}], Loss: {loss_total.item():.4f}')
    if stop_training:
        print(f"Early stopping at Autoencoder_epoch {epoch+1}, loss reached {loss_total.item():.6f}")
        break 

Autoencoder_epoch [1], Loss: 0.0017
Autoencoder_epoch [2], Loss: 0.0001
Early stopping at Autoencoder_epoch 2, loss reached 0.000127


In [48]:
# Training classic classifier
shared_features, num_list = extract_features(model, data_loader, device)
shared_features = shared_features.cpu()

target_AE = []
for i in num_list:
    i = i.numpy()
    y = target_dict[(i[0])]
    target_AE.append(y)    
target_AE = np.array(target_AE)

RF = RandomForestClassifier(n_estimators=200, max_depth=50, random_state=0)
SVC_model = SVC(gamma='auto', probability=True)
Ada = AdaBoostClassifier(n_estimators=200, algorithm="SAMME.R", learning_rate=1.0, random_state=0)
GNB = GaussianNB()

classifier_list = [RF, SVC_model, Ada, GNB]
classifier_name = ['RF', 'SVC','Ada','GNB']
time_list = []

for n, classifier in enumerate(classifier_list):

    kfold = StratifiedKFold(n_splits=4, shuffle=True, random_state=0)
    all_test_y = []
    all_test_p = []
    all_test_prob = []
    for train_idx,test_idx in kfold.split(shared_features, target_AE):
                train_x,test_x = shared_features[train_idx],shared_features[test_idx]
                train_y,test_y = target_AE[train_idx],target_AE[test_idx]
                classifier.fit(train_x,train_y)
                test_p = classifier.predict(test_x)
                test_prob = classifier.predict_proba(test_x)[:,0]
                all_test_y.append(test_y)
                all_test_p.append(test_p)
                all_test_prob.append(test_prob)
    all_test_y = np.concatenate(all_test_y)
    all_test_p = np.concatenate(all_test_p)
    all_test_prob = np.concatenate(all_test_prob)
    acc = metrics.accuracy_score(all_test_y, all_test_p)
    auc = roc_auc_score(1 - all_test_y, all_test_prob)
    f1 = f1_score(all_test_y, all_test_p, pos_label=0)

    print(
        "{} - Accuracy: {:.4f}, AUC: {:.4f}, F1: {:.4f}".format(
            classifier_name[n],
            acc,
            auc,
            f1
        )
    )

shared_features shape before reshape: torch.Size([1696, 1])
shared_features shape after reshape: torch.Size([53, 32])
RF - Accuracy: 0.8679, AUC: 0.8222, F1: 0.3636
SVC - Accuracy: 0.8491, AUC: 0.7194, F1: 0.0000
Ada - Accuracy: 0.8679, AUC: 0.8042, F1: 0.5882
GNB - Accuracy: 0.6415, AUC: 0.7778, F1: 0.4242
